# Kaggle histopathology introduction
This notebook is an introduction to the data challenge of out of distribution classification of histopathology patches. It also serves as a baseline for the code and the model.

If you have any questions, feel free to contact me at [leo.fillioux@centralesupelec.fr](mailto:leo.fillioux@centralesupelec.fr).

In [29]:
import h5py
import torch
import random
import numpy as np
import pandas as pd
import torchmetrics
import matplotlib.pyplot as plt
import torchvision.transforms as transforms
from tqdm.notebook import tqdm
from torch.utils.data import Dataset, DataLoader

In [30]:
TRAIN_IMAGES_PATH = 'train.h5'
VAL_IMAGES_PATH = 'val.h5'
TEST_IMAGES_PATH = 'test.h5'
SEED = 0

In [31]:
torch.random.manual_seed(SEED)
random.seed(SEED)

## 2. Building a baseline model
The baseline model consists of extracting DINOv2 embeddings and linear probing.

In [32]:
BATCH_SIZE = 128

### 2.1. Baseline dataset
We start by creating the model to read and process the data. For this simple model we also use another dataset with the preprocessed embeddings to avoid recomputing the same embeddings each time.

In [33]:
import h5py
import torch
import numpy as np
from torch.utils.data import Dataset
#! GPT to speed up I/O
class BaselineDataset(Dataset):
    def __init__(self, dataset_path, preprocessing, mode):
        super(BaselineDataset, self).__init__()
        self.dataset_path = dataset_path
        self.preprocessing = preprocessing
        self.mode = mode
        
        # Open the file once and keep it open for reading
        self.hdf = h5py.File(self.dataset_path, 'r')
        self.image_ids = list(self.hdf.keys())

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        img_id = self.image_ids[idx]
        img = self.hdf[img_id]['img'][()]  # Read the image directly as a NumPy array
        label = self.hdf[img_id]['label'][()] if self.mode == 'train' else -1  # Use -1 for test mode

        return self.preprocessing(torch.tensor(img)).float(), label

    def __del__(self):
        # Ensure the file is closed when the dataset is deleted
        if hasattr(self, 'hdf') and self.hdf is not None:
            self.hdf.close()

In [34]:
def precompute(dataloader, model, device):
    xs, ys = [], []
    for x, y in tqdm(dataloader, leave=False):
        with torch.no_grad():
            xs.append(model(x.to(device)).detach().cpu().numpy())
        ys.append(y.numpy())
    xs = np.vstack(xs)
    ys = np.hstack(ys)
    return torch.tensor(xs), torch.tensor(ys)

In [35]:
class PrecomputedDataset(Dataset):
    def __init__(self, features, labels):
        super(PrecomputedDataset, self).__init__()
        self.features = features
        self.labels = labels.unsqueeze(-1)
    
    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx].float()

In [36]:
preprocessing = transforms.Resize((98, 98),antialias=None)
train_dataset = BaselineDataset(TRAIN_IMAGES_PATH, preprocessing, 'train')
val_dataset = BaselineDataset(VAL_IMAGES_PATH, preprocessing, 'train')

In [37]:
train_dataloader = DataLoader(train_dataset, shuffle=True, batch_size=BATCH_SIZE,num_workers=2)
val_dataloader = DataLoader(val_dataset, shuffle=False, batch_size=BATCH_SIZE,num_workers=2)

### 2.2. Building the models and precomputing the features

In [38]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Working on {device}.')

Working on cuda.


In [39]:
feature_extractor = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14').to(device)
feature_extractor.eval()
linear_probing = torch.nn.Sequential(torch.nn.Linear(feature_extractor.num_features, 1),
                                     torch.nn.Sigmoid()).to(device)

Using cache found in /raid/home/bournez_pie/.cache/torch/hub/facebookresearch_dinov2_main


In [40]:
x,y=precompute(train_dataloader, feature_extractor, device)
train_dataset = PrecomputedDataset(features=x, labels=y)
x,y=precompute(val_dataloader, feature_extractor, device)
val_dataset = PrecomputedDataset(features=x, labels=y)

  0%|          | 0/782 [00:00<?, ?it/s]

  0%|          | 0/273 [00:00<?, ?it/s]

In [41]:
train_dataloader = DataLoader(train_dataset, shuffle=True, batch_size=BATCH_SIZE,num_workers=2)
val_dataloader = DataLoader(val_dataset, shuffle=False, batch_size=BATCH_SIZE,num_workers=2)

## 3. Training the model

In [42]:
OPTIMIZER = 'Adam'
OPTIMIZER_PARAMS = {'lr': 0.001}
LOSS = 'BCELoss'
METRIC = 'Accuracy'
NUM_EPOCHS = 100
PATIENCE = 10

In [43]:
optimizer = getattr(torch.optim, OPTIMIZER)(linear_probing.parameters(), **OPTIMIZER_PARAMS)
criterion = getattr(torch.nn, LOSS)()
metric = getattr(torchmetrics, METRIC)('binary')
min_loss, best_epoch = float('inf'), 0

In [44]:
for epoch in range(NUM_EPOCHS):
    linear_probing.train()
    train_metrics, train_losses = [], []
    for train_x, train_y in tqdm(train_dataloader, leave=False):
        optimizer.zero_grad()
        train_pred = linear_probing(train_x.to(device))
        loss = criterion(train_pred, train_y.to(device))
        loss.backward()
        optimizer.step()
        train_losses.extend([loss.item()]*len(train_y))
        train_metric = metric(train_pred.cpu(), train_y.int().cpu())
        train_metrics.extend([train_metric.item()]*len(train_y))
    print(f'Epoch train [{epoch+1}/{NUM_EPOCHS}] | Loss {np.mean(train_losses):.4f} | Metric {np.mean(train_metrics):.4f}')

    linear_probing.eval()
    val_metrics, val_losses = [], []
    for val_x, val_y in tqdm(val_dataloader, leave=False):
        with torch.no_grad():
            val_pred = linear_probing(val_x.to(device))
        loss = criterion(val_pred, val_y.to(device))
        val_losses.extend([loss.item()]*len(val_y))
        val_metric = metric(val_pred.cpu(), val_y.int().cpu())
        val_metrics.extend([val_metric.item()]*len(val_y))
    print(f'Epoch valid [{epoch+1}/{NUM_EPOCHS}] | Loss {np.mean(val_losses):.4f} | Metric {np.mean(val_metrics):.4f}')

    if np.mean(val_losses) < min_loss:
        mean_val_loss = np.mean(val_losses)
        print(f'New best loss {min_loss:.4f} -> {mean_val_loss:.4f}')
        min_loss = mean_val_loss
        best_epoch = epoch
        torch.save(linear_probing.state_dict(), 'best_model.pth')

    if epoch - best_epoch == PATIENCE:
        break

  0%|          | 0/782 [00:00<?, ?it/s]

Epoch train [1/100] | Loss 0.2032 | Metric 0.9231


  0%|          | 0/273 [00:00<?, ?it/s]

Epoch valid [1/100] | Loss 0.3540 | Metric 0.8470
New best loss inf -> 0.3540


  0%|          | 0/782 [00:00<?, ?it/s]

...

Epoch valid [20/100] | Loss 0.3411 | Metric 0.8655


  0%|          | 0/782 [00:00<?, ?it/s]

Epoch train [21/100] | Loss 0.1372 | Metric 0.9486


  0%|          | 0/273 [00:00<?, ?it/s]

Epoch valid [21/100] | Loss 0.3346 | Metric 0.8668


## 4. Making the final prediction

To create a solutions file, you need to generate a CSV with 2 columns.
- **ID**: containing the ID of the image
- **Pred**: with the predicted class (**threshold the prediction to get either 0 or 1**)

In [45]:
linear_probing.load_state_dict(torch.load('best_model.pth', weights_only=True))
linear_probing.eval()
linear_probing.to(device)
prediction_dict = {}

/raid/home/bournez_pie/mva_geom/mva_geom_24/DLMI/project/venv/lib/python3.8/site-packages/torch/_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()


In [46]:
with h5py.File(TEST_IMAGES_PATH, 'r') as hdf:
    test_ids = list(hdf.keys())

In [47]:
test_dataset=BaselineDataset(TEST_IMAGES_PATH,preprocessing=preprocessing,mode="test")
test_dataloader = DataLoader(test_dataset, shuffle=False, batch_size=BATCH_SIZE)
print("start")
x,y=precompute(test_dataloader, feature_extractor, device)
test_dataset = PrecomputedDataset(features=x, labels=y)
test_dataloader = DataLoader(test_dataset, shuffle=False, batch_size=BATCH_SIZE,num_workers=2)
predictions = []

for test_x, _ in tqdm(test_dataloader, leave=False):
    with torch.no_grad():
        test_pred = linear_probing(test_x.to(device))
    predictions.append(test_pred.cpu().numpy())
predictions = np.vstack(predictions)


start


  0%|          | 0/665 [00:00<?, ?it/s]

  0%|          | 0/665 [00:00<?, ?it/s]

In [52]:
(predictions>0.5).squeeze().astype(int)

array([0, 1, 1, ..., 1, 1, 1])

In [53]:
solutions_data = {'ID': [], 'Pred': []}
solutions_data["Pred"]=(predictions>0.5).squeeze().astype(int)
solutions_data["ID"]=test_ids
solutions_data = pd.DataFrame(solutions_data).set_index('ID')
solutions_data.to_csv('baseline.csv')

In [ ]:
# solutions_data = {'ID': [], 'Pred': []}
# with h5py.File(TEST_IMAGES_PATH, 'r') as hdf:
#     for test_id in tqdm(test_ids):
#         img = preprocessing(torch.tensor(np.array(hdf.get(test_id).get('img')))).unsqueeze(0).float()
#         pred = linear_probing(feature_extractor(img.to(device))).detach().cpu()
#         solutions_data['ID'].append(int(test_id))
#         solutions_data['Pred'].append(int(pred.item() > 0.5))
# solutions_data = pd.DataFrame(solutions_data).set_index('ID')
# solutions_data.to_csv('baseline.csv')

  0%|          | 0/85054 [00:00<?, ?it/s]

KeyboardInterrupt: 